# Create mosaic with rects non-colliding, of different size and randomly placed

## Sources:
- [Stackoverflow](https://stackoverflow.com/questions/4373741/how-can-i-randomly-place-several-non-colliding-rects)

In [2]:
import random
from random import randint
random.seed()

class Point(object):
    def __init__(self, x, y):
        self.x, self.y = x, y

    @staticmethod
    def from_point(other):
        return Point(other.x, other.y)

class Rect(object):
    def __init__(self, x1, y1, x2, y2):
        minx, maxx = (x1,x2) if x1 < x2 else (x2,x1)
        miny, maxy = (y1,y2) if y1 < y2 else (y2,y1)
        self.min, self.max = Point(minx, miny), Point(maxx, maxy)

    @staticmethod
    def from_points(p1, p2):
        return Rect(p1.x, p1.y, p2.x, p2.y)

    width  = property(lambda self: self.max.x - self.min.x)
    height = property(lambda self: self.max.y - self.min.y)

plus_or_minus = lambda v: v * [-1, 1][(randint(0, 100) % 2)]  # equal chance +/-1


NUM_RECTS = 20
REGION = Rect(0, 0, 640, 480)

def quadsect(rect, factor):
    """ Subdivide given rectangle into four non-overlapping rectangles.
        'factor' is an integer representing the proportion of the width or
        height the deviatation from the center of the rectangle allowed.
    """
    # pick a point in the interior of given rectangle
    w, h = rect.width, rect.height  # cache properties
    center = Point(rect.min.x + (w // 2), rect.min.y + (h // 2))
    delta_x = plus_or_minus(randint(0, w // factor))
    delta_y = plus_or_minus(randint(0, h // factor))
    interior = Point(center.x + delta_x, center.y + delta_y)

    # create rectangles from the interior point and the corners of the outer one
    return [Rect(interior.x, interior.y, rect.min.x, rect.min.y),
            Rect(interior.x, interior.y, rect.max.x, rect.min.y),
            Rect(interior.x, interior.y, rect.max.x, rect.max.y),
            Rect(interior.x, interior.y, rect.min.x, rect.max.y)]

def square_subregion(rect):
    """ Return a square rectangle centered within the given rectangle """
    w, h = rect.width, rect.height  # cache properties
    if w < h:
        offset = (h - w) // 2
        return Rect(rect.min.x, rect.min.y+offset,
                    rect.max.x, rect.min.y+offset+w)
    else:
        offset = (w - h) // 2
        return Rect(rect.min.x+offset, rect.min.y,
                    rect.min.x+offset+h, rect.max.y)

# call quadsect() until at least the number of rects wanted has been generated
rects = [REGION]   # seed output list
while len(rects) <= NUM_RECTS:
    rects = [subrect for rect in rects
                        for subrect in quadsect(rect, 3)]

random.shuffle(rects)  # mix them up
sample = random.sample(rects, NUM_RECTS)  # select the desired number
print('%d out of the %d rectangles selected' % (NUM_RECTS, len(rects)))

#################################################
# extra credit - create an image file showing results

from PIL import Image, ImageDraw

def gray(v): return tuple(int(v*255) for _ in range(3))

BLACK, DARK_GRAY, GRAY = gray(0), gray(.25), gray(.5)
LIGHT_GRAY, WHITE = gray(.75), gray(1)
RED, GREEN, BLUE = (255, 0, 0), (0, 255, 0), (0, 0, 255)
CYAN, MAGENTA, YELLOW = (0, 255, 255), (255, 0, 255), (255, 255, 0)
BACKGR, SQUARE_COLOR, RECT_COLOR = (245, 245, 87), (255, 73, 73), (37, 182, 249)

imgx, imgy = REGION.max.x + 1, REGION.max.y + 1
image = Image.new("RGB", (imgx, imgy), BACKGR)  # create color image
draw = ImageDraw.Draw(image)

def draw_rect(rect, fill=None, outline=WHITE):
    draw.rectangle([(rect.min.x, rect.min.y), (rect.max.x, rect.max.y)],
                   fill=fill, outline=outline)

# first draw outlines of all the non-overlapping rectanges generated
for rect in rects:
    draw_rect(rect, outline=LIGHT_GRAY)

# then draw the random sample of them selected
for rect in sample:
    draw_rect(rect, fill=RECT_COLOR, outline=WHITE)

# and lastly convert those into squares and re-draw them in another color
for rect in sample:
    draw_rect(square_subregion(rect), fill=SQUARE_COLOR, outline=WHITE)

filename = 'square_quadsections.png'
image.save(filename, "PNG")
print(repr(filename), 'output image saved')

20 out of the 64 rectangles selected
'square_quadsections.png' output image saved


In [3]:
import itertools
from datetime import date
import pathlib
import random
import cv2
import numpy as np
from numba import jit
from IPython import display
import imageio
import ray
from skimage import exposure

In [77]:
# set project directory & directory where to save mosaic results images
project_dir = pathlib.Path('/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/winter-olympics')
project_name = "diff_rects"
mosaic_images_dir = project_dir / f"mosaic_results_{project_name}"
mosaic_images_dir.mkdir(parents=True, exist_ok=True)

# # get image paths
# img_dir = pathlib.Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/avatar_dd/db_flower")
# EXTENSIONS = {'.png', '.jpg', '.jpeg'}
# img_paths = [str(image_path) for image_path in img_dir.glob('**/*') if image_path.suffix in EXTENSIONS]

# # shuffle images
# random.shuffle(img_paths)

# choose number of tasks
nb_tasks = 4

# # split list into n chunks (n = number of multiprocessing tasks)
# img_paths_chunks = np.array_split(img_paths, nb_tasks)

# get photo to recreate
photo_path = "/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/winter_olympics/to_recreate/chloe-kim.jpeg"
print(photo_path)
photo = cv2.imread(photo_path)


# set grid caracteristics
number_rows = 100
number_cols = 100


# read and get first image shape as reference
# ref_img = cv2.imread(img_paths_chunks[0][0])
# ref_img = cv2.imread(img_paths[0])
ref_img = photo
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape
ref_img_height, ref_img_width = ref_img_height//25, ref_img_width//25

# set output grid image shape
out_img_height = ref_img_height * number_rows
out_img_width = ref_img_width * number_cols
out_img_channel = ref_img_channel

photo = cv2.resize(photo, (out_img_width, out_img_height))


imgs = [cv2.resize(cv2.imread(img_path), (ref_img_width, ref_img_height)) for img_path in img_paths[:number_rows*number_cols]]
# imgs = [cv2.resize(cv2.cvtColor(cv2.imread(img_path, cv2.IMREAD_UNCHANGED), cv2.COLOR_BGRA2BGR), (ref_img_width, ref_img_height)) for img_path in img_paths[:number_rows*number_cols]]

# set image matrix canvas
img_matrix = np.zeros((out_img_height, out_img_width, out_img_channel), np.uint8)
# img_matrix.fill(255)


# shutdown ray
ray.shutdown()

# start ray
ray.init(log_to_driver=True)


# @jit(nopython=True)
# def create_mosaic_same_img(photo, img_paths, number_rows, number_cols, grid_range):
@ray.remote
def create_mosaic_same_img(photo, number_rows, number_cols, grid_range):
# def create_mosaic_same_img(photo, imgs, number_rows, number_cols, grid_range):
    """
    Function to recreate given photo with photos mosaic
    Arguments:
    * grid_range: to assign grid range for each worker
    """
    # # set grid caracteristics
    # number_rows = 100
    # number_cols = 100
    # total_grid_cells = number_rows * number_cols

    # # get the required number of image paths
    # img_paths = img_paths[:total_grid_cells]

    # # read and get first image shape as reference
    # ref_img = cv2.imread(img_paths[0])
    # ref_img_height, ref_img_width, ref_img_channel = ref_img.shape
    # ref_img_height, ref_img_width = ref_img_height//1, ref_img_height//1


    # # set output grid image shape
    # out_img_height = ref_img_height * number_rows + margin_y * (number_rows - 1)
    # out_img_width = ref_img_width * number_cols + margin_x * (number_cols - 1)
    # out_img_channel = ref_img_channel

    # # set image matrix canvas
    # img_matrix = np.zeros((out_img_height, out_img_width, out_img_channel), np.uint8)
    # img_matrix.fill(255)

    # photo = cv2.resize(photo, (out_img_width, out_img_height))

    # create empty dictionary to track grid positions and their corresponding image numpy arrays
    imgs_dict = {}

    # set margins between rects (#toDo: to scale it with image size)
    margin_x = 2
    margin_y = 2

    # loop through grid cell positions
    grid_positions = itertools.product(range(*grid_range), range(number_rows))
    for (x_i, y_i) in grid_positions:
        print(x_i, y_i)

        # get region of interest in given photo to recreate
        x = x_i * (ref_img_width)
        y = y_i * (ref_img_height)
        roi = photo[y:y + ref_img_height, x:x + ref_img_width, :]

        roi_rect = Rect(0, 0, ref_img_width, ref_img_height)

        # roi_mean_bgr = np.array(cv2.mean(roi)[:-1]).astype(np.uint8)

        # loop twice to place new rects on top of other new rects
        for i in np.linspace(1, 2, 2):
            roi_rects = quadsect(roi_rect, 3)
            roi = roi.copy()
            for rect in roi_rects:
                if rect.min.y <= rect.max.y:
                    y1, y2 = rect.min.y, rect.max.y
                else:
                    y2, y1 = rect.min.y, rect.max.y

                if rect.min.x <= rect.max.x:
                    x1, x2 = rect.min.x, rect.max.x
                else:
                    x2, x1 = rect.min.x, rect.max.x

                
                # new_roi[rect.min.y:rect.max.y, rect.min.x:rect.max.x, :] = np.array((12, 34, 55), dtype=np.uint8)
                # roi[y1:y2, x1:x2, :] = (np.random.randint(0,255,(3))).astype(np.uint8)
                
                # get new rect in roi
                new_rect = roi[y1:y2-int(margin_y*i), x1:x2-int(margin_x*i), :]

                # get mean bgr color in new rect
                new_rect_mean = np.array(cv2.mean(new_rect//i)[:-1]).astype(np.uint8)

                # fill rect with mean bgr color of new rect
                new_rect[:,:,:] = new_rect_mean

                # color margins
                new_rect[y2:y2 + margin_y, x2:x2 + margin_x, :].fill(0)

                # # convert roi and img to hsv
                # roi_rect_hsv = cv2.cvtColor(roi_rect, cv2.COLOR_BGR2HSV)

            # # change value with random float between 0 and 1
            # # For HSV, hue range is [0,179], saturation range is [0,255], and value range is [0,255].
            # roi_rect_hsv[:,:,1] = np.random.randint(0,255)
            
            # # convert image back to bgr
            # roi_rect = cv2.cvtColor(roi_rect_hsv, cv2.COLOR_HSV2BGR)

            # # update rect in roi
            # roi[y1:y2, x1:x2, :] = roi_rect

        # # inititiate tracker for best match to that roi
        # best_match = np.inf
        # best_match_index = 0

        # # # list of paths for images with a different shape than the reference one
        # # diff_shapes_img_paths = []


        # # get an image with required shape and that best match photo to recreate
        # # for img_idx, img_path in enumerate(img_paths[:number_rows*number_cols]):
        # for img_idx, img in enumerate(imgs[:number_rows*number_cols]):

        #     # img = cv2.resize(cv2.imread(img_path), (ref_img_width, ref_img_height)) 

        #    # compare image with roi
        #     total_sum = np.sum(np.abs(roi - img)) # toDO: try cv2.compareHist(H1, H2, method)
        #     # total_sum = np.sum(np.abs(roi - img)) / (ref_img_width*ref_img_height) / 255 # toDO: try cv2.compareHist(H1, H2, method)

        #     #   # get the similarity values
        #     #   structural_sim = structural_sim(img_a, img_b)
        #     #   pixel_sim = pixel_sim(img_a, img_b)
        #     #   sift_sim = sift_sim(img_a, img_b)
        #     #   emd = earth_movers_distance(img_a, img_b)
        #     #   print(structural_sim, pixel_sim, sift_sim, emd)

        #     if total_sum < best_match:
        #         best_match = total_sum
        #         best_match_index = img_idx

        # # take out image path from list
        # # best_match_path = img_paths.pop(best_match_index)
        # # best_match_path = img_paths.pop(0)
        # # best_match_path = img_paths[best_match_index]

        # # img = cv2.resize(cv2.imread(best_match_path), (ref_img_width, ref_img_height)) 
        # img = imgs[best_match_index]

        # # convert roi and img to hsv
        # roi_hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        # img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        
        # # # compute mean hue of roi
        # # roi_mean_hue = np.mean(roi_hsv[:,:,0])

        # # # apply mean hue of roi to img
        # # img_hsv[:,:,0] = roi_mean_hue

        # # # convert image back to bgr
        # # img = cv2.cvtColor(img_hsv, cv2.COLOR_HSV2BGR)

        # # define lower and uppper limits of what we call "black-ish"
        # sensitivity = 5
        # lower=np.array([0,0,0])
        # upper=np.array([255,255,sensitivity])

        # # mask image to only select "black-ish"
        # mask=cv2.inRange(img_hsv,lower,upper)

        # # # Change image to red where we found brown
        # # image[mask>0]=(0,0,255)

        # # copy image to prevent read-only error (ToDo: check why read-only)
        # img = img.copy()

        # # change image to mean bgr of roi where image is "black-ish"
        # img[mask==0] = roi_mean_bgr

        # # blend brightness from new colorized image and original image
        # new_img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        # alpha = 0.7
        # beta = 1 - alpha
        # new_img_hsv[:,:,-1] = cv2.addWeighted(new_img_hsv[:,:,-1],alpha, img_hsv[:,:,-1],beta, 0)

        # # convert image back to bgr
        # img = cv2.cvtColor(new_img_hsv, cv2.COLOR_HSV2BGR)

        # # img = match_histograms(img, roi)
        # # img = exposure.match_histograms(img, roi, multichannel=True)
    
        # # # copy image otherwise worker seems to share same image
        # # new_img = img.copy()
        # # new_img = exposure.match_histograms(img, roi, multichannel=True).astype(np.uint8)
        # # alpha = 0.9
        # # beta = 1 - alpha
        # # new_img = cv2.addWeighted(new_img, alpha, img, beta, 0)

        # # imgs_dict[(best_match_path,y,x)] = img
        # imgs_dict[(best_match_index,y,x)] = img
        imgs_dict[("best_match_index",y,x)] = roi
 
    return imgs_dict


# start tasks in parallel
result_ids = []
# chunk_nb = 0
for i in range(0, number_cols, number_cols//nb_tasks): # set grid range for each worker

    # recreate given photo with photos mosaic
    # result_ids.append(create_mosaic_same_img.remote(photo, img_paths_chunks[chunk_nb].tolist(), number_rows, number_cols, (i, i+number_cols//nb_tasks)))
    # result_ids.append(create_mosaic_same_img.remote(photo, img_paths, number_rows, number_cols, (i, i+number_cols//nb_tasks)))
    # result_ids.append(create_mosaic_same_img.remote(photo, imgs, number_rows, number_cols, (i, i+number_cols//nb_tasks)))
    result_ids.append(create_mosaic_same_img.remote(photo, number_rows, number_cols, (i, i+number_cols//nb_tasks)))

    # # increment chunk number
    # chunk_nb += 1
    
# wait for the tasks to complete and retrieve the results
results = ray.get(result_ids)

# create list to store best match path and xy grid positions
best_match_paths = []
# best_match_indexes = []

# combine results and save image matrix
for result in results:
    for key, cell_image in result.items():
        # best_match_index, best_degree, y, x = key
        best_match_path, y, x = key
        # best_match_index, y, x = key
        img_matrix[y:y+ref_img_height, x:x+ref_img_width, :] = cell_image

        # add result metadata to list
        best_match_paths.append(key)
        # best_match_indexes.append(key)
        # best_match_indexes.append((img_paths[best_match_index], y, x))

# get current date
today = date.today().strftime("%Y%m%d")

# set output image & metadata paths
out_img_path = str(mosaic_images_dir / f"matrix_photo_{number_rows*number_cols}_{today}.jpg")
# out_metadata_path = str(mosaic_images_dir / f"best_match_paths_{number_rows*number_cols}_{today}.npy")
out_metadata_path = str(mosaic_images_dir / f"best_match_paths_{number_rows*number_cols}_{today}.npy")

# save output image & metadata
cv2.imwrite(out_img_path, img_matrix)
np.save(out_metadata_path, np.array(best_match_paths))
# np.save(out_metadata_path, np.array(best_match_indexes))


 # shutdown ray
ray.shutdown()

/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/winter_olympics/to_recreate/chloe-kim.jpeg
(create_mosaic_same_img pid=53791) 0 0
(create_mosaic_same_img pid=53791) 0 1
(create_mosaic_same_img pid=53791) 0 2
(create_mosaic_same_img pid=53791) 0 3
(create_mosaic_same_img pid=53791) 0 4
(create_mosaic_same_img pid=53791) 0 5
(create_mosaic_same_img pid=53791) 0 6
(create_mosaic_same_img pid=53791) 0 7
(create_mosaic_same_img pid=53791) 0 8
(create_mosaic_same_img pid=53791) 0 9
(create_mosaic_same_img pid=53791) 0 10
(create_mosaic_same_img pid=53791) 0 11
(create_mosaic_same_img pid=53791) 0 12
(create_mosaic_same_img pid=53791) 0 13
(create_mosaic_same_img pid=53791) 0 14
(create_mosaic_same_img pid=53791) 0 15
(create_mosaic_same_img pid=53791) 0 16
(create_mosaic_same_img pid=53791) 0 17
(create_mosaic_same_img pid=53791) 0 18
(create_mosaic_same_img pid=53791) 0 19
(create_mosaic_same_img pid=53791) 0 20
(create_mosaic_same_img pid=53791) 0 21
(create_mosa

2022-03-07 10:06:43,104	WARNING worker.py:462 -- The driver may not be able to keep up with the stdout/stderr of the workers. To avoid forwarding logs to the driver, use 'ray.init(log_to_driver=False)'.



(create_mosaic_same_img pid=53791) 5 3
(create_mosaic_same_img pid=53791) 5 4
(create_mosaic_same_img pid=53791) 5 5
(create_mosaic_same_img pid=53791) 5 6
(create_mosaic_same_img pid=53791) 5 7
(create_mosaic_same_img pid=53791) 5 8
(create_mosaic_same_img pid=53791) 5 9
(create_mosaic_same_img pid=53791) 5 10
(create_mosaic_same_img pid=53791) 5 11
(create_mosaic_same_img pid=53791) 5 12
(create_mosaic_same_img pid=53791) 5 13
(create_mosaic_same_img pid=53791) 5 14
(create_mosaic_same_img pid=53791) 5 15
(create_mosaic_same_img pid=53791) 5 16
(create_mosaic_same_img pid=53791) 5 17
(create_mosaic_same_img pid=53791) 5 18
(create_mosaic_same_img pid=53791) 5 19
(create_mosaic_same_img pid=53791) 5 20
(create_mosaic_same_img pid=53791) 5 21
(create_mosaic_same_img pid=53791) 5 22
(create_mosaic_same_img pid=53791) 5 23
(create_mosaic_same_img pid=53791) 5 24
(create_mosaic_same_img pid=53791) 5 25
(create_mosaic_same_img pid=53791) 5 26
(create_mosaic_same_img pid=53791) 5 27
(creat